In [0]:
# Load the dataset
storage_account_name = "amazondatalake60304948"
storage_account_key = "t8pg68f8Wo84sAYKeZ6k8wDm/X2WOYuUcRLMwcRo2UTs/lyzH8ok/m2c/hJPg7LZ3cRp/kD3y/Gp+ASt4JvYPw=="
spark.conf.set(
 f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
 storage_account_key
)

In [0]:
# Show schema + types + rows, columns, sample rows
features_path = f"abfss://curated@{storage_account_name}.dfs.core.windows.net/features_v1/"
df = spark.read.parquet(features_path)

print("Rows:", df.count())
print("Cols:", len(df.columns))
df.printSchema()
display(df.limit(5))

Rows: 40498248
Cols: 11
root
 |-- asin: string (nullable = true)
 |-- title: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- reviewerID: string (nullable = true)
 |-- overall: double (nullable = true)
 |-- summary: string (nullable = true)
 |-- reviewText: string (nullable = true)
 |-- helpful: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- reviewTime: string (nullable = true)
 |-- review_year: integer (nullable = true)



asin title brand price reviewerID overall summary reviewText helpful reviewTime review_year 1400599997 Barnes & Noble NOOK ebook reader (WiFi + 3G)[B&W] null 78.71 A19N3S7CBSU6O7 5.0 Nook eReader I will continue to update this review as I discover new features of the Nook, so stay tuned folks!*****************************************************************************Updated 02/08/2010*****************************************************************************I've been using my Nook with the newest update (v 1.2) for a few days now. This is the second update to the Nook since launch. All I can say is THANK YOU Barnes & Noble!The LCD screen improvements alone are GREAT! With better looking icons, more responsiveness, and a sleeker appearance -- it makes me think of that McDonald's commercial: I'm lovin' it!Several of the well documented issues have been fixed, such as losing bookmarks due to powering off the unit -- which as it turns out was the most cited problem for Nook bashers and haters alike. Nook haters will also be disappointed that the battery life has been optimized too.All-in-all if you've been waiting for the minor 1.0 issues to be fixed prior to purchasing the Nook you will not be disappointed in this newest version. And if you have been considering switching from the Kindle you would be well advised to do so at this point.*****************************************************************************Updated 01/31/2010*****************************************************************************This device is clearly the best option on the market today, with it's LCD touchscreen, Wi-fi, memory expansion capabilities, and so, so much more. Let's "wade in the weeds" and look at some of the features in detail:LCD TouchscreenSo far this is the only mainstream reader to offer a LCD screen of any kind. The Coverflow-like book cover listing is beautiful, elegant, and functional. With the newest update it is uber fast, intuitive, and responsive. Page turns are super easy with the touchscreen, all you have to do is swipe the screen with one finger when it's in standby mode and the page turns. In addition, it's easier for me to navigate books via the LCD than textually. Seeing the cover is a really nice feature, since books are designed to pique your interest by the cover. It just seems like a good cover sells a book, and seeing that cover in crisp LCD sells me from time to time.The keyboard is nice, any time a text field comes up you will have the keyboard pop up on queue. It's fast, responsive, and best of all doesn't look like a 1970's era fax machine keypad like some of the older readers on the market.Wi-fiThe built in Wi-fi is one of the great features of this device. Modern 3G speeds are pretty fast, but as most of us know that have smartphones, Wi-fi is about 100 times faster, especially when one tries to download large things, or a lot of content. The Wi-fi on this device is no exception... it's fast and super easy to set up. The on screen instructions are sooo clear to follow. I couldn't imagine only having 3G these days, like when you are in an area that doesn't have 3G coverage, you can still use the Nook over Wi-fi. This is truly one of the killer features of the Nook. You can go into a B&N; store and Wi-fi to your hearts content, but then come home and do the same. Hotels offering free Wi-fi are the Nooks biach.Memory ExpansionThe Nook has a Micro-SD card slot which accepts up to 16GB Micro-SD cards, this allows you to upgrade your memory indefinitely. Another advantage would be to organize your books. And books on the card are never going to get lost if your reader dies -- without having to re-download from the cloud. Plus with the new B&N; desktop reader you can move your books from the Nook to a laptop/netbook on the fly, and what if the worst should happen and your eReader dies? You still have those books on the card.AppearanceThis is a biggy for me as it is for most people... I don't tend to buy or use thing

In [0]:
# Type Check
from pyspark.sql.types import StringType, DoubleType, IntegerType

schema_dict = {field.name: field.dataType for field in df.schema.fields}

print("asin type:", schema_dict.get("asin"))
print("reviewerID type:", schema_dict.get("reviewerID"))
print("overall type:", schema_dict.get("overall"))
print("reviewText type:", schema_dict.get("reviewText"))

asin type: StringType()
reviewerID type: StringType()
overall type: DoubleType()
reviewText type: StringType()


In [0]:
# Confirm No Unexpected Types
for field in df.schema.fields:
    print(f"{field.name}: {field.dataType}")

asin: StringType()
title: StringType()
brand: StringType()
price: DoubleType()
reviewerID: StringType()
overall: DoubleType()
summary: StringType()
reviewText: StringType()
helpful: ArrayType(IntegerType(), True)
reviewTime: StringType()
review_year: IntegerType()


In [0]:
# Check for missing or empty values
from pyspark.sql import functions as F

required_cols = ["asin", "reviewerID", "overall", "reviewText"]
missing = [c for c in required_cols if c not in df.columns]
print("Missing required cols:", missing)

checks = df.select(
    F.col("asin"),
    F.col("reviewerID"),
    F.col("overall"),
    F.col("reviewText")
).agg(
    F.sum(F.col("asin").isNull().cast("int")).alias("asin_null"),
    F.sum(F.col("reviewerID").isNull().cast("int")).alias("reviewerID_null"),
    F.sum(F.col("overall").isNull().cast("int")).alias("overall_null"),
    F.sum(
        (F.col("reviewText").isNull() | (F.length(F.trim(F.col("reviewText"))) == 0)).cast("int")
    ).alias("reviewText_empty_or_null")
)

display(checks)

Missing required cols: []


asin_null,reviewerID_null,overall_null,reviewText_empty_or_null
0,0,0,0


In [0]:
# Count ratings
rating_counts = df.groupBy("overall").count().orderBy("overall")
display(rating_counts)

overall,count
1.0,2606688
2.0,1969344
3.0,3410544
4.0,8320632
5.0,24191040


Databricks visualization. Run in Databricks to view.

In [0]:
# Review Length Distribution
from pyspark.sql import functions as F

length_df = df.withColumn("review_length", F.length(F.col("reviewText")))

display(length_df.select("review_length"))

review_length
7775
1775
234
1456
315
702
108
240
1952
262


Databricks visualization. Run in Databricks to view.

In [0]:
# Optional (Cleaner Visualization)
length_filtered = length_df.filter(F.col("review_length") < 2000)
display(length_filtered.select("review_length"))

review_length
1775
234
1456
315
702
108
240
1952
262
995


Databricks visualization. Run in Databricks to view.

In [0]:
# Percentage column (Optional / bonus)
from pyspark.sql import functions as F

total = df.count()
rating_pct = rating_counts.withColumn("percent", F.round(F.col("count")/F.lit(total)*100, 2))
display(rating_pct)

overall,count,percent
1.0,2606688,6.44
2.0,1969344,4.86
3.0,3410544,8.42
4.0,8320632,20.55
5.0,24191040,59.73


In [0]:
# Rating vs Review Length (Optional / bonus)
length_df = df.withColumn("review_len", F.length("reviewText"))

length_by_rating = length_df.groupBy("overall") \
    .agg(F.round(F.avg("review_len"),2).alias("avg_review_length")) \
    .orderBy("overall")

display(length_by_rating)

overall,avg_review_length
1.0,656.58
2.0,777.03
3.0,780.96
4.0,778.95
5.0,550.79


Databricks visualization. Run in Databricks to view.

In [0]:
# Detect Rating Imbalance Ratio (Optional / bonus)
rating_pct
max_pct = rating_pct.agg(F.max("percent")).collect()[0][0]
min_pct = rating_pct.agg(F.min("percent")).collect()[0][0]
print("Imbalance ratio:", round(max_pct/min_pct,2))

Imbalance ratio: 12.29


In [0]:
# Rating Distribution Over Time (Optional / bonus)
rating_year = df.groupBy("review_year") \
                .agg(F.round(F.avg("overall"),2).alias("avg_rating"),
                     F.count("*").alias("review_count")) \
                .orderBy("review_year")

display(rating_year)

review_year,avg_rating,review_count
1999,4.39,1728
2000,4.32,19608
2001,4.19,38616
2002,4.11,55560
2003,4.01,85128
2004,3.88,123816
2005,3.91,231312
2006,3.99,370728
2007,4.14,863040
2008,4.15,1195992


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
# B
print("review_year" in df.columns)

True


In [0]:
# Calculate Total Rows and Sampling Fraction
total_rows = df_sampled.count()
sample_size = 300000
fraction = sample_size / total_rows

print("Total rows:", total_rows)
print("Sampling fraction:", fraction)

Total rows: 40498248
Sampling fraction: 0.007407727860227435


In [0]:
# Create Stratified Sample by Year
from pyspark.sql import functions as F

# Get distinct years
years = [row["review_year"] for row in df.select("review_year").distinct().collect()]

# Create fraction dictionary (same fraction for each year)
fractions = {year: fraction for year in years}

# Stratified sample
df_sampled = df.sampleBy("review_year", fractions=fractions, seed=42)

print("Sampled rows:", df_sampled.count())

# A stratified sampling strategy was applied using review_year as the stratification column. A proportional fraction (0.0074) was applied to each year to generate approximately 300,000 rows. This preserves the temporal distribution of the dataset and reduces potential language drift bias.

Sampled rows: 300073


In [0]:
df_sampled.printSchema()

root
 |-- asin: string (nullable = true)
 |-- title: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- reviewerID: string (nullable = true)
 |-- overall: double (nullable = true)
 |-- summary: string (nullable = true)
 |-- reviewText: string (nullable = true)
 |-- helpful: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- reviewTime: string (nullable = true)
 |-- review_year: integer (nullable = true)



In [0]:
# Compare rating distribution
orig_dist = df.groupBy("overall").count().orderBy("overall").withColumnRenamed("count","orig_count")
samp_dist = df_sampled.groupBy("overall").count().orderBy("overall").withColumnRenamed("count","sample_count")

display(orig_dist.join(samp_dist, on="overall", how="outer").orderBy("overall"))

# The sampled dataset preserves both the original schema and rating distribution. A comparison between the full dataset and the 300,000-row stratified sample shows that rating proportions remain consistent. This confirms that the sampling process maintained representativeness while reducing dataset size for efficient downstream feature engineering

overall,orig_count,sample_count
1.0,2606688,19247
2.0,1969344,14510
3.0,3410544,25154
4.0,8320632,61693
5.0,24191040,179469


In [0]:
# Create Clean Text Column
from pyspark.sql import functions as F

df_feat = df_sampled.withColumn(
    "clean_text",
    F.lower(F.col("reviewText"))
)

display(df_feat.select("reviewText", "clean_text").limit(5))
# The review text was converted to lowercase to ensure consistent token representation. This prevents the same word (e.g., “Great” and “great”) from being treated as separate tokens during feature extraction.

reviewText,clean_text
"I have used both Western Digital Green 1TB and Seagate Barracuda 7200.11 1TB in DLINK 323 NAS device for about a month. These are my experiences... Seagate didn't need firmware update.Packaging:WD was packaged very well with cushion padding where as Seagate was packed really lousy. I had ordered 2 drives in each brand and one of the seagate boxes looked like it was opened already (may be it was human error while packaging; I didn't bother).Noise:Seagate is noisier than WD but it is much smoother than WD. Meaning, WD produced lot of hard clicking sounds that really irritated me (both drives; may be just the two I received were defective). Clicking sounds from Seagate felt normal to me (and it was muffled).Heat:Seagate got much hotter than WD. I figured WD didn't produce as much as heat because it uses ""intelliseek"" that runs the motor at optimal/slower speeds rather than the advertised 7200rpm.Speed:I couldn't compare speed as I was running WD in RAID1 and Seagate as Standard drives.Overall, I am very happy with Seagate mainly because of its much smoother operation and brand reputation.Hope this review helps!","i have used both western digital green 1tb and seagate barracuda 7200.11 1tb in dlink 323 nas device for about a month. these are my experiences... seagate didn't need firmware update.packaging:wd was packaged very well with cushion padding where as seagate was packed really lousy. i had ordered 2 drives in each brand and one of the seagate boxes looked like it was opened already (may be it was human error while packaging; i didn't bother).noise:seagate is noisier than wd but it is much smoother than wd. meaning, wd produced lot of hard clicking sounds that really irritated me (both drives; may be just the two i received were defective). clicking sounds from seagate felt normal to me (and it was muffled).heat:seagate got much hotter than wd. i figured wd didn't produce as much as heat because it uses ""intelliseek"" that runs the motor at optimal/slower speeds rather than the advertised 7200rpm.speed:i couldn't compare speed as i was running wd in raid1 and seagate as standard drives.overall, i am very happy with seagate mainly because of its much smoother operation and brand reputation.hope this review helps!"
This product performs as described. I have no complaints. I bought it to use multiple computers from my internet connection using wired connections. So far it's worked flawlessly.,this product performs as described. i have no complaints. i bought it to use multiple computers from my internet connection using wired connections. so far it's worked flawlessly.
"Works with our camcorder, have not had any problems with it at all. Not sure if its fast or slow as far as transfer speeds, but it works great for me.","works with our camcorder, have not had any problems with it at all. not sure if its fast or slow as far as transfer speeds, but it works great for me."
"This lens is very fast and very sharp! It is sharper than my older Sigma 50mm macro f/2.8. I used it at a concert in low light without flash, and the pictures are remarquable! I shot at f/1.4 ISO 800 1/160sec and I am only using a D50 and the results are perfect!","this lens is very fast and very sharp! it is sharper than my older sigma 50mm macro f/2.8. i used it at a concert in low light without flash, and the pictures are remarquable! i shot at f/1.4 iso 800 1/160sec and i am only using a d50 and the results are perfect!"
"2nd review. After having this lens and using it for a while I had to add a few important points here. It is still great quality and sharp but using it without a tripod turned out to be difficult especially at longer focal lengths. There is no VR and unless you are always using one for longer shots you might have issues. Actually I am selling this and checking out the Nikon 70-300 VR. I can't handle this with longer focal lengths unless its always on a tripod which keeps it in my bag more often than it should be. T

In [0]:
# Tokenizer
from pyspark.ml.feature import Tokenizer

tokenizer = Tokenizer(
    inputCol="reviewText",
    outputCol="tokens"
)

df_tokens = tokenizer.transform(df_sampled)

display(df_tokens.select("reviewText", "tokens").limit(5))

# A Tokenizer transformer was applied to split review text into individual word tokens. Tokenization is the first step in converting raw text into structured numerical features suitable for machine learning models.

reviewText,tokens
"I have used both Western Digital Green 1TB and Seagate Barracuda 7200.11 1TB in DLINK 323 NAS device for about a month. These are my experiences... Seagate didn't need firmware update.Packaging:WD was packaged very well with cushion padding where as Seagate was packed really lousy. I had ordered 2 drives in each brand and one of the seagate boxes looked like it was opened already (may be it was human error while packaging; I didn't bother).Noise:Seagate is noisier than WD but it is much smoother than WD. Meaning, WD produced lot of hard clicking sounds that really irritated me (both drives; may be just the two I received were defective). Clicking sounds from Seagate felt normal to me (and it was muffled).Heat:Seagate got much hotter than WD. I figured WD didn't produce as much as heat because it uses ""intelliseek"" that runs the motor at optimal/slower speeds rather than the advertised 7200rpm.Speed:I couldn't compare speed as I was running WD in RAID1 and Seagate as Standard drives.Overall, I am very happy with Seagate mainly because of its much smoother operation and brand reputation.Hope this review helps!","List(i, have, used, both, western, digital, green, 1tb, and, seagate, barracuda, 7200.11, 1tb, in, dlink, 323, nas, device, for, about, a, month., these, are, my, experiences..., seagate, didn't, need, firmware, update.packaging:wd, was, packaged, very, well, with, cushion, padding, where, as, seagate, was, packed, really, lousy., i, had, ordered, 2, drives, in, each, brand, and, one, of, the, seagate, boxes, looked, like, it, was, opened, already, (may, be, it, was, human, error, while, packaging;, i, didn't, bother).noise:seagate, is, noisier, than, wd, but, it, is, much, smoother, than, wd., meaning,, wd, produced, lot, of, hard, clicking, sounds, that, really, irritated, me, (both, drives;, may, be, just, the, two, i, received, were, defective)., clicking, sounds, from, seagate, felt, normal, to, me, (and, it, was, muffled).heat:seagate, got, much, hotter, than, wd., i, figured, wd, didn't, produce, as, much, as, heat, because, it, uses, ""intelliseek"", that, runs, the, motor, at, optimal/slower, speeds, rather, than, the, advertised, 7200rpm.speed:i, couldn't, compare, speed, as, i, was, running, wd, in, raid1, and, seagate, as, standard, drives.overall,, i, am, very, happy, with, seagate, mainly, because, of, its, much, smoother, operation, and, brand, reputation.hope, this, review, helps!)"
This product performs as described. I have no complaints. I bought it to use multiple computers from my internet connection using wired connections. So far it's worked flawlessly.,"List(this, product, performs, as, described., i, have, no, complaints., i, bought, it, to, use, multiple, computers, from, my, internet, connection, using, wired, connections., so, far, it's, worked, flawlessly.)"
"Works with our camcorder, have not had any problems with it at all. Not sure if its fast or slow as far as transfer speeds, but it works great for me.","List(works, with, our, camcorder,, have, not, had, any, problems, with, it, at, all., , not, sure, if, its, fast, or, slow, as, far, as, transfer, speeds,, but, it, works, great, for, me.)"
"This lens is very fast and very sharp! It is sharper than my older Sigma 50mm macro f/2.8. I used it at a concert in low light without flash, and the pictures are remarquable! I shot at f/1.4 ISO 800 1/160sec and I am only using a D50 and the results are perfect!","List(this, lens, is, very, fast, and, very, sharp!, it, is, sharper, than, my, older, sigma, 50mm, macro, f/2.8., i, used, it, at, a, concert, in, low, light, without, flash,, and, the, pictures, are, remarquable!, i, shot, at, f/1.4, iso, 800, 1/160sec, and, i, am, only, using, a, d50, and, the, results, are, perfect!)"
"2nd review. After having this lens and using it for a while I had to add a few important points here. It is still great quality and sharp but using it without a tripod turned out to be difficult especi

In [0]:
# Remove Stopwords
from pyspark.ml.feature import StopWordsRemover

remover = StopWordsRemover(
    inputCol="tokens",
    outputCol="filtered_tokens"
)

df_filtered = remover.transform(df_tokens)

display(df_filtered.select("tokens", "filtered_tokens").limit(5))

# Stopword removal was applied to reduce noise by removing very common words that do not contribute meaningful information. This helps improve feature quality by focusing on more informative terms.

tokens,filtered_tokens
"List(i, have, used, both, western, digital, green, 1tb, and, seagate, barracuda, 7200.11, 1tb, in, dlink, 323, nas, device, for, about, a, month., these, are, my, experiences..., seagate, didn't, need, firmware, update.packaging:wd, was, packaged, very, well, with, cushion, padding, where, as, seagate, was, packed, really, lousy., i, had, ordered, 2, drives, in, each, brand, and, one, of, the, seagate, boxes, looked, like, it, was, opened, already, (may, be, it, was, human, error, while, packaging;, i, didn't, bother).noise:seagate, is, noisier, than, wd, but, it, is, much, smoother, than, wd., meaning,, wd, produced, lot, of, hard, clicking, sounds, that, really, irritated, me, (both, drives;, may, be, just, the, two, i, received, were, defective)., clicking, sounds, from, seagate, felt, normal, to, me, (and, it, was, muffled).heat:seagate, got, much, hotter, than, wd., i, figured, wd, didn't, produce, as, much, as, heat, because, it, uses, ""intelliseek"", that, runs, the, motor, at, optimal/slower, speeds, rather, than, the, advertised, 7200rpm.speed:i, couldn't, compare, speed, as, i, was, running, wd, in, raid1, and, seagate, as, standard, drives.overall,, i, am, very, happy, with, seagate, mainly, because, of, its, much, smoother, operation, and, brand, reputation.hope, this, review, helps!)","List(used, western, digital, green, 1tb, seagate, barracuda, 7200.11, 1tb, dlink, 323, nas, device, month., experiences..., seagate, need, firmware, update.packaging:wd, packaged, well, cushion, padding, seagate, packed, really, lousy., ordered, 2, drives, brand, one, seagate, boxes, looked, like, opened, already, (may, human, error, packaging;, bother).noise:seagate, noisier, wd, much, smoother, wd., meaning,, wd, produced, lot, hard, clicking, sounds, really, irritated, (both, drives;, may, two, received, defective)., clicking, sounds, seagate, felt, normal, (and, muffled).heat:seagate, got, much, hotter, wd., figured, wd, produce, much, heat, uses, ""intelliseek"", runs, motor, optimal/slower, speeds, rather, advertised, 7200rpm.speed:i, compare, speed, running, wd, raid1, seagate, standard, drives.overall,, happy, seagate, mainly, much, smoother, operation, brand, reputation.hope, review, helps!)"
"List(this, product, performs, as, described., i, have, no, complaints., i, bought, it, to, use, multiple, computers, from, my, internet, connection, using, wired, connections., so, far, it's, worked, flawlessly.)","List(product, performs, described., complaints., bought, use, multiple, computers, internet, connection, using, wired, connections., far, worked, flawlessly.)"
"List(works, with, our, camcorder,, have, not, had, any, problems, with, it, at, all., , not, sure, if, its, fast, or, slow, as, far, as, transfer, speeds,, but, it, works, great, for, me.)","List(works, camcorder,, problems, all., , sure, fast, slow, far, transfer, speeds,, works, great, me.)"
"List(this, lens, is, very, fast, and, very, sharp!, it, is, sharper, than, my, older, sigma, 50mm, macro, f/2.8., i, used, it, at, a, concert, in, low, light, without, flash,, and, the, pictures, are, remarquable!, i, shot, at, f/1.4, iso, 800, 1/160sec, and, i, am, only, using, a, d50, and, the, results, are, perfect!)","List(lens, fast, sharp!, sharper, older, sigma, 50mm, macro, f/2.8., used, concert, low, light, without, flash,, pictures, remarquable!, shot, f/1.4, iso, 800, 1/160sec, using, d50, results, perfect!)"
"List(2nd, review., after, having, this, lens, and, using, it, for, a, while, i, had, to, add, a, few, important, points, here., , it, is, still, great, quality, and, sharp, but, using, it, without, a, tripod, turned, out, to, be, difficult, especially, at, longer, focal, lengths., there, is, no, vr, and, unless, you, are, always, using, one, for, longer, shots, you, might, have, issues., actually, i, am, selling, this, and, checking, out, the, nikon, 70-300, vr., i, can't, handle, this, with, longer, focal, lengths, unless, its, alway

In [0]:
# Fit CountVectorizer
from pyspark.ml.feature import CountVectorizer

cv = CountVectorizer(
    inputCol="filtered_tokens",
    outputCol="tf_features",
    vocabSize=5000,
    minDF=5
)

cv_model = cv.fit(df_filtered)
df_tf = cv_model.transform(df_filtered)

display(df_tf.select("filtered_tokens", "tf_features").limit(5))

filtered_tokens,tf_features
"List(used, western, digital, green, 1tb, seagate, barracuda, 7200.11, 1tb, dlink, 323, nas, device, month., experiences..., seagate, need, firmware, update.packaging:wd, packaged, well, cushion, padding, seagate, packed, really, lousy., ordered, 2, drives, brand, one, seagate, boxes, looked, like, opened, already, (may, human, error, packaging;, bother).noise:seagate, noisier, wd, much, smoother, wd., meaning,, wd, produced, lot, hard, clicking, sounds, really, irritated, (both, drives;, may, two, received, defective)., clicking, sounds, seagate, felt, normal, (and, muffled).heat:seagate, got, much, hotter, wd., figured, wd, produce, much, heat, uses, ""intelliseek"", runs, motor, optimal/slower, speeds, rather, advertised, 7200rpm.speed:i, compare, speed, running, wd, raid1, seagate, standard, drives.overall,, happy, seagate, mainly, much, smoother, operation, brand, reputation.hope, review, helps!)","Map(vectorType -> sparse, length -> 5000, indices -> List(1, 3, 8, 10, 13, 22, 24, 31, 33, 49, 55, 70, 75, 99, 131, 199, 200, 211, 214, 284, 289, 304, 312, 345, 355, 382, 410, 497, 519, 525, 526, 663, 685, 907, 912, 998, 1000, 1046, 1119, 1154, 1250, 1256, 1308, 1400, 1626, 1697, 1881, 1942, 2037, 2157, 2452, 2463, 2470, 2489, 2648, 2748, 2962, 4070, 4576, 4669, 4863), values -> List(1.0, 1.0, 2.0, 4.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 2.0, 2.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 4.0, 7.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 2.0, 1.0, 1.0, 2.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 2.0))"
"List(product, performs, described., complaints., bought, use, multiple, computers, internet, connection, using, wired, connections., far, worked, flawlessly.)","Map(vectorType -> sparse, length -> 5000, indices -> List(2, 15, 17, 38, 91, 101, 269, 307, 529, 845, 864, 1411, 1897, 2152, 2810, 3420), values -> List(1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0))"
"List(works, camcorder,, problems, all., , sure, fast, slow, far, transfer, speeds,, works, great, me.)","Map(vectorType -> sparse, length -> 5000, indices -> List(0, 6, 7, 84, 101, 169, 188, 241, 422, 543, 629, 4901), values -> List(1.0, 1.0, 2.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0))"
"List(lens, fast, sharp!, sharper, older, sigma, 50mm, macro, f/2.8., used, concert, low, light, without, flash,, pictures, remarquable!, shot, f/1.4, iso, 800, 1/160sec, using, d50, results, perfect!)","Map(vectorType -> sparse, length -> 5000, indices -> List(15, 22, 46, 68, 88, 182, 188, 237, 370, 961, 1040, 1220, 1334, 2117, 2357, 2469, 2795, 3618, 4315), values -> List(1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0))"
"List(2nd, review., lens, using, add, important, points, here., , still, great, quality, sharp, using, without, tripod, turned, difficult, especially, longer, focal, lengths., vr, unless, always, using, one, longer, shots, might, issues., actually, selling, checking, nikon, 70-300, vr., handle, longer, focal, lengths, unless, always, tripod, keeps, bag, often, be., , great, lens,, bokah, nice,, fairly, compact,, great, color,, quick, focusing,, bit, heavy, close, weight, 80-200, vr's., really, happy, heading, nikon, 70-300, see, great, reviews, deserved, substitute, 80-200, now., want, save, price, 80-200, look, used, one, b&h, adorama., saved, $250, getting, used, like, new.)","Map(vectorType -> sparse, length -> 5000, indices -> List(0, 1, 3, 6, 8, 15, 18, 22, 29, 35, 36, 46, 57, 62, 68, 105, 109, 131, 135, 141, 167, 178, 247, 260, 295, 333, 349, 373, 409, 414, 417, 456, 484, 501, 503, 521, 535, 577, 578, 630, 682, 739, 747, 766, 891, 974, 1019, 1286, 1310, 1362, 1388, 1482, 1573, 1635, 2256, 2496, 2650, 2687, 2900, 3701), values -> List(2.0, 2.0, 1.0, 4.0, 1.0, 3.0, 1.0, 2.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 2.0, 1.0, 1.0, 1.0, 1.0, 1.0, 3.0, 1.0, 1.0, 2.0

In [0]:
# Save vocabulary size (quick check)
print("Vocabulary size:", len(cv_model.vocabulary))
print("Sample vocab:", cv_model.vocabulary[:20])

# CountVectorizer was used to transform filtered tokens into numeric term-frequency vectors. A vocabulary cap (vocabSize=5000) was applied to limit dimensionality, and rare words were filtered out (minDF=5) to reduce sparsity and noise. These TF features provide a structured representation of text suitable for downstream modeling.

Vocabulary size: 5000
Sample vocab: ['', 'one', 'use', 'like', 'get', 'good', 'great', 'works', 'really', 'also', 'much', 'camera', '-', 'well', 'even', 'using', 'sound', 'bought', 'quality', 'case']


In [0]:
# Compute TF-IDF
from pyspark.ml.feature import IDF

idf = IDF(
    inputCol="tf_features",
    outputCol="tfidf_features"
)

idf_model = idf.fit(df_tf)
df_tfidf = idf_model.transform(df_tf)

display(df_tfidf.select("tf_features", "tfidf_features").limit(5))

# TF-IDF was computed from term-frequency vectors to emphasize informative words and downweight words that appear frequently across many reviews. Compared to raw TF, TF-IDF typically improves text feature quality for classification and regression tasks by capturing term importance rather than raw frequency.

tf_features,tfidf_features
"Map(vectorType -> sparse, length -> 5000, indices -> List(1, 3, 8, 10, 13, 22, 24, 31, 33, 49, 55, 70, 75, 99, 131, 199, 200, 211, 214, 284, 289, 304, 312, 345, 355, 382, 410, 497, 519, 525, 526, 663, 685, 907, 912, 998, 1000, 1046, 1119, 1154, 1250, 1256, 1308, 1400, 1626, 1697, 1881, 1942, 2037, 2157, 2452, 2463, 2470, 2489, 2648, 2748, 2962, 4070, 4576, 4669, 4863), values -> List(1.0, 1.0, 2.0, 4.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 2.0, 2.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 4.0, 7.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 2.0, 1.0, 1.0, 2.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 2.0))","Map(vectorType -> sparse, length -> 5000, indices -> List(1, 3, 8, 10, 13, 22, 24, 31, 33, 49, 55, 70, 75, 99, 131, 199, 200, 211, 214, 284, 289, 304, 312, 345, 355, 382, 410, 497, 519, 525, 526, 663, 685, 907, 912, 998, 1000, 1046, 1119, 1154, 1250, 1256, 1308, 1400, 1626, 1697, 1881, 1942, 2037, 2157, 2452, 2463, 2470, 2489, 2648, 2748, 2962, 4070, 4576, 4669, 4863), values -> List(1.3420558003061553, 1.4598197543600653, 3.846341759592789, 7.724380057542601, 1.8936323944722622, 2.037227698383952, 2.1067396884738376, 2.210610358924367, 2.317585901763247, 2.5302396996939094, 2.6598789947805144, 2.899129232046648, 2.647130946785875, 2.880234798507992, 2.9543473269166545, 3.4947572683354995, 3.3403250437642296, 3.4644502844379654, 3.3789716513150942, 3.5211651049145436, 3.5369170312652334, 3.576645011834719, 3.9481669717945773, 3.735521204277633, 3.698193745431458, 7.532437709633038, 7.802088003328961, 3.9764806422452095, 4.022739002613641, 4.042426523278253, 4.386654568080757, 4.254459895345155, 4.29457988913458, 4.579342251483026, 4.527887000661402, 4.777927515440061, 20.35678611943754, 36.638290412911076, 4.782283095479716, 4.988702203852732, 4.89029261930912, 5.047468796776855, 4.9336598484242495, 4.9935921891469235, 5.256103523252153, 5.258017399434437, 5.444177732535557, 5.885668662900484, 5.6139628157099555, 5.493044822740644, 11.517776954750515, 5.675043329943677, 5.694613426137774, 11.655160650760209, 5.898913889650505, 5.853882584901758, 5.895284121599926, 6.545871687741076, 6.961808095716535, 6.529797862910014, 13.196280232573665))"
"Map(vectorType -> sparse, length -> 5000, indices -> List(2, 15, 17, 38, 91, 101, 269, 307, 529, 845, 864, 1411, 1897, 2152, 2810, 3420), values -> List(1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0))","Map(vectorType -> sparse, length -> 5000, indices -> List(2, 15, 17, 38, 91, 101, 269, 307, 529, 845, 864, 1411, 1897, 2152, 2810, 3420), values -> List(1.417453357988436, 1.9814511609265404, 1.8993241059410286, 2.3408336332711968, 2.7088616222310415, 2.797621120436021, 3.6233574690443784, 3.804583735476918, 4.065799234740536, 4.530660072843638, 4.633931510873939, 4.998998936159371, 5.310454050707481, 5.470679673281064, 5.830972075381226, 6.117522285799323))"
"Map(vectorType -> sparse, length -> 5000, indices -> List(0, 6, 7, 84, 101, 169, 188, 241, 422, 543, 629, 4901), values -> List(1.0, 1.0, 2.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0))","Map(vectorType -> sparse, length -> 5000, indices -> List(0, 6, 7, 84, 101, 169, 188, 241, 422, 543, 629, 4901), values -> List(0.8743956136805712, 1.4963718692753576, 3.3356669572052704, 2.69355937004253, 2.797621120436021, 3.1424391515779013, 3.2394599321276, 3.3786790968247344, 3.7773421186537557, 4.245836138369348, 4.241436455458868, 6.622832728877204))"
"Map(vectorType -> sparse, length -> 5000, indices -> List(15, 22, 46, 68, 88, 182, 188, 237, 370, 961, 1040, 1220, 1334, 2117, 2357, 2469, 2795, 3618, 4315), values -> List(1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0))","Map(vectorType -> sparse, length -> 5000, indices -> List(15, 22, 46, 68, 88, 182, 188, 237, 370, 961, 1040, 1220, 1334, 2117, 2357, 2469, 2795, 3618, 4315), values -> List(1.98145116

In [0]:
# Create the pipeline
from pyspark.ml import Pipeline
from pyspark.ml.feature import Tokenizer, StopWordsRemover, CountVectorizer, IDF

tokenizer = Tokenizer(inputCol="reviewText", outputCol="tokens")
remover = StopWordsRemover(inputCol="tokens", outputCol="filtered_tokens")
cv = CountVectorizer(inputCol="filtered_tokens", outputCol="tf_features", vocabSize=5000, minDF=5)
idf = IDF(inputCol="tf_features", outputCol="tfidf_features")

pipeline = Pipeline(stages=[tokenizer, remover, cv, idf])
pipeline_model = pipeline.fit(df_sampled)

df_pipeline_out = pipeline_model.transform(df_sampled)

display(df_pipeline_out.select("reviewText", "tokens", "filtered_tokens", "tfidf_features").limit(5))

# A Spark ML Pipeline was created to standardize text feature engineering into a single reproducible workflow. The pipeline includes tokenization, stopword removal, term-frequency vectorization (CountVectorizer), and TF-IDF transformation (IDF). This ensures consistent preprocessing and supports scalable reuse on new datasets.

reviewText,tokens,filtered_tokens,tfidf_features
"I have used both Western Digital Green 1TB and Seagate Barracuda 7200.11 1TB in DLINK 323 NAS device for about a month. These are my experiences... Seagate didn't need firmware update.Packaging:WD was packaged very well with cushion padding where as Seagate was packed really lousy. I had ordered 2 drives in each brand and one of the seagate boxes looked like it was opened already (may be it was human error while packaging; I didn't bother).Noise:Seagate is noisier than WD but it is much smoother than WD. Meaning, WD produced lot of hard clicking sounds that really irritated me (both drives; may be just the two I received were defective). Clicking sounds from Seagate felt normal to me (and it was muffled).Heat:Seagate got much hotter than WD. I figured WD didn't produce as much as heat because it uses ""intelliseek"" that runs the motor at optimal/slower speeds rather than the advertised 7200rpm.Speed:I couldn't compare speed as I was running WD in RAID1 and Seagate as Standard drives.Overall, I am very happy with Seagate mainly because of its much smoother operation and brand reputation.Hope this review helps!","List(i, have, used, both, western, digital, green, 1tb, and, seagate, barracuda, 7200.11, 1tb, in, dlink, 323, nas, device, for, about, a, month., these, are, my, experiences..., seagate, didn't, need, firmware, update.packaging:wd, was, packaged, very, well, with, cushion, padding, where, as, seagate, was, packed, really, lousy., i, had, ordered, 2, drives, in, each, brand, and, one, of, the, seagate, boxes, looked, like, it, was, opened, already, (may, be, it, was, human, error, while, packaging;, i, didn't, bother).noise:seagate, is, noisier, than, wd, but, it, is, much, smoother, than, wd., meaning,, wd, produced, lot, of, hard, clicking, sounds, that, really, irritated, me, (both, drives;, may, be, just, the, two, i, received, were, defective)., clicking, sounds, from, seagate, felt, normal, to, me, (and, it, was, muffled).heat:seagate, got, much, hotter, than, wd., i, figured, wd, didn't, produce, as, much, as, heat, because, it, uses, ""intelliseek"", that, runs, the, motor, at, optimal/slower, speeds, rather, than, the, advertised, 7200rpm.speed:i, couldn't, compare, speed, as, i, was, running, wd, in, raid1, and, seagate, as, standard, drives.overall,, i, am, very, happy, with, seagate, mainly, because, of, its, much, smoother, operation, and, brand, reputation.hope, this, review, helps!)","List(used, western, digital, green, 1tb, seagate, barracuda, 7200.11, 1tb, dlink, 323, nas, device, month., experiences..., seagate, need, firmware, update.packaging:wd, packaged, well, cushion, padding, seagate, packed, really, lousy., ordered, 2, drives, brand, one, seagate, boxes, looked, like, opened, already, (may, human, error, packaging;, bother).noise:seagate, noisier, wd, much, smoother, wd., meaning,, wd, produced, lot, hard, clicking, sounds, really, irritated, (both, drives;, may, two, received, defective)., clicking, sounds, seagate, felt, normal, (and, muffled).heat:seagate, got, much, hotter, wd., figured, wd, produce, much, heat, uses, ""intelliseek"", runs, motor, optimal/slower, speeds, rather, advertised, 7200rpm.speed:i, compare, speed, running, wd, raid1, seagate, standard, drives.overall,, happy, seagate, mainly, much, smoother, operation, brand, reputation.hope, review, helps!)","Map(vectorType -> sparse, length -> 5000, indices -> List(1, 3, 8, 10, 13, 22, 24, 31, 33, 49, 55, 70, 75, 99, 131, 199, 200, 211, 214, 284, 289, 304, 312, 345, 355, 382, 410, 497, 519, 525, 526, 663, 685, 907, 912, 998, 1000, 1046, 1119, 1156, 1250, 1256, 1308, 1400, 1625, 1697, 1881, 1942, 2037, 2157, 2453, 2463, 2470, 2489, 2648, 2748, 2962, 4071, 4577, 4672, 4870), values -> List(1.3420558003061553, 1.4598197543600653, 3.846341759592789, 7.724380057542601, 1.8936323944722622, 2.037227698383952, 2.1067396884738376, 2.210610358924367, 2.317585901763247, 2.5302396996939094, 2

In [0]:
# Write output to curated layer
out_path = f"abfss://curated@{storage_account_name}.dfs.core.windows.net/features_v1/"
df_pipeline_out.write.mode("overwrite").parquet(out_path)

print("Saved to:", out_path)

Saved to: abfss://curated@amazondatalake60304948.dfs.core.windows.net/features_v1/


In [0]:
# Quick verification (required sanity check)
check = spark.read.parquet(out_path)
print("Rows saved:", check.count())
display(check.select("reviewText","tfidf_features").limit(5))

Rows saved: 300140


reviewText,tfidf_features
"I have used both Western Digital Green 1TB and Seagate Barracuda 7200.11 1TB in DLINK 323 NAS device for about a month. These are my experiences... Seagate didn't need firmware update.Packaging:WD was packaged very well with cushion padding where as Seagate was packed really lousy. I had ordered 2 drives in each brand and one of the seagate boxes looked like it was opened already (may be it was human error while packaging; I didn't bother).Noise:Seagate is noisier than WD but it is much smoother than WD. Meaning, WD produced lot of hard clicking sounds that really irritated me (both drives; may be just the two I received were defective). Clicking sounds from Seagate felt normal to me (and it was muffled).Heat:Seagate got much hotter than WD. I figured WD didn't produce as much as heat because it uses ""intelliseek"" that runs the motor at optimal/slower speeds rather than the advertised 7200rpm.Speed:I couldn't compare speed as I was running WD in RAID1 and Seagate as Standard drives.Overall, I am very happy with Seagate mainly because of its much smoother operation and brand reputation.Hope this review helps!","Map(vectorType -> sparse, length -> 5000, indices -> List(1, 3, 8, 10, 13, 22, 24, 31, 33, 49, 55, 70, 75, 99, 131, 199, 200, 211, 214, 284, 289, 304, 312, 345, 355, 382, 410, 497, 519, 525, 526, 663, 685, 907, 912, 998, 1000, 1046, 1119, 1156, 1250, 1256, 1308, 1400, 1625, 1697, 1881, 1942, 2037, 2157, 2453, 2463, 2470, 2489, 2648, 2748, 2962, 4071, 4577, 4672, 4870), values -> List(1.3420558003061553, 1.4598197543600653, 3.846341759592789, 7.724380057542601, 1.8936323944722622, 2.037227698383952, 2.1067396884738376, 2.210610358924367, 2.317585901763247, 2.5302396996939094, 2.6598789947805144, 2.899129232046648, 2.647130946785875, 2.880234798507992, 2.9543473269166545, 3.4947572683354995, 3.3403250437642296, 3.4644502844379654, 3.3789716513150942, 3.5211651049145436, 3.5369170312652334, 3.576645011834719, 3.9481669717945773, 3.735521204277633, 3.698193745431458, 7.532437709633038, 7.802088003328961, 3.9764806422452095, 4.022739002613641, 4.042426523278253, 4.386654568080757, 4.254459895345155, 4.29457988913458, 4.579342251483026, 4.527887000661402, 4.777927515440061, 20.35678611943754, 36.638290412911076, 4.782283095479716, 4.988702203852732, 4.89029261930912, 5.047468796776855, 4.9336598484242495, 4.9935921891469235, 5.256103523252153, 5.258017399434437, 5.444177732535557, 5.885668662900484, 5.6139628157099555, 5.493044822740644, 11.517776954750515, 5.675043329943677, 5.694613426137774, 11.655160650760209, 5.898913889650505, 5.853882584901758, 5.895284121599926, 6.545871687741076, 6.961808095716535, 6.529797862910014, 13.196280232573665))"
This product performs as described. I have no complaints. I bought it to use multiple computers from my internet connection using wired connections. So far it's worked flawlessly.,"Map(vectorType -> sparse, length -> 5000, indices -> List(2, 15, 17, 38, 91, 101, 269, 307, 529, 845, 864, 1411, 1897, 2152, 2810, 3419), values -> List(1.417453357988436, 1.9814511609265404, 1.8993241059410286, 2.3408336332711968, 2.7088616222310415, 2.797621120436021, 3.6233574690443784, 3.804583735476918, 4.065799234740536, 4.530660072843638, 4.633931510873939, 4.998998936159371, 5.310454050707481, 5.470679673281064, 5.830972075381226, 6.117522285799323))"
"Works with our camcorder, have not had any problems with it at all. Not sure if its fast or slow as far as transfer speeds, but it works great for me.","Map(vectorType -> sparse, length -> 5000, indices -> List(0, 6, 7, 84, 101, 169, 188, 241, 422, 543, 629, 4900), values -> List(0.8743956136805712, 1.4963718692753576, 3.3356669572052704, 2.69355937004253, 2.797621120436021, 3.1424391515779013, 3.2394599321276, 3.3786790968247344, 3.7773421186537557, 4.245836138369348, 4.241436455458868, 6.622832728877204))"
"This lens is very fast and very sharp! It is sharper than my older Sigma 50mm macro f/2.8. I used it at a concert 